# 02 — Fit and tune random forests

Compare the same five feature variants as the original logistic workflow. Each forest is tuned using the **same candidate settings and the same forward event-purged folds**. Selection minimizes row-weighted validation log loss, fixed in advance. ROC-AUC, Brier score, average precision and accuracy are also recorded.

The October 3 outer validation is a diagnostic check, not the tuning target. October 4–5 outcomes never determine settings, features, threshold or importance rankings.

In [1]:
# Locate this add-on folder whether Jupyter starts here or in 02_notebooks.
from pathlib import Path
import os, sys, json
from IPython.display import display, Markdown, Image

here = Path.cwd().resolve()
helper_dir = next((p for base in [here, *here.parents]
                   for p in [base, base / '02_notebooks']
                   if (p / 'rf_common.py').is_file()), None)
if helper_dir is None:
    raise FileNotFoundError('Keep rf_common.py beside these notebooks and start Jupyter in the extracted add-on.')
if str(helper_dir) not in sys.path:
    sys.path.insert(0, str(helper_dir))
PACKAGE_ROOT = helper_dir.parent
import rf_common as rf
print('Add-on folder:', PACKAGE_ROOT)

Add-on folder: /workspace/scratch/2f27ad4cbed5/output/wallet_full_analysis


In [2]:
# Use the exact run created in notebook 01; no original results are overwritten.
OUTPUT_BASE = Path(os.environ.get('WALLET_RF_OUTPUT_BASE', str(PACKAGE_ROOT / '04_results' / 'random_forest'))).expanduser().resolve()
override = os.environ.get('WALLET_RF_RUN_DIR')
saved_run = Path(json.loads((OUTPUT_BASE / 'latest_run.json').read_text())['run_dir'])
RUN_DIR = Path(override).expanduser().resolve() if override else (saved_run if saved_run.is_absolute() and saved_run.exists() else OUTPUT_BASE / saved_run.name)
CONFIG = json.loads((RUN_DIR / 'run_config.json').read_text())
print('Run:', RUN_DIR)
print('Evaluation status:', CONFIG['evaluation_status'])

Run: /workspace/scratch/2f27ad4cbed5/output/wallet_full_analysis/04_results/random_forest/full_data_run
Evaluation status: Previously inspected test period; exploratory reuse


## 1. Inspect forward cross-validation before fitting

With the supplied week and original split there are three full development days (September 30–October 2). Therefore the default is **two daily forward folds**, not ordinary five-fold shuffled CV:

| Fold | Training entries | Validation entries |
|---|---|---|
| 1 | Before October 1 | October 1 |
| 2 | Before October 2 | October 2 |

Training also requires recorded resolution before the fold starts. Validation outcomes must resolve before the fold ends. Entire validation event groups are purged from training, including earlier members. Folds that are too small are reported; fewer than two usable folds raises an error rather than silently changing methods.

Using a longer period would permit more informative forward folds.

In [3]:
import pandas as pd
frames = rf.load_frames(RUN_DIR)
all_keys = pd.read_parquet(RUN_DIR / 'all_entry_keys.parquet')
folds, fold_audit = rf.forward_event_folds(frames['train'], all_keys, CONFIG)
display(fold_audit)
print('Usable forward folds:', len(folds))

   fold  ...                             validation_keys_sha256
0     1  ...  6e0ff765ef6346d07df91fe7b447b42700071a55fbfa3a...
1     2  ...  da7ca28ea90cec3d9a8ab576756a01367b181fcf242b39...

[2 rows x 10 columns]
Usable forward folds: 2


## 2. Tune, check validation and refit

For every candidate/fold, a new median imputer is fitted **only on that fold's training rows**. No scaling is needed for the forest. The logistic comparator uses the original fixed `C=1`, median imputation and training-fitted scaling, with missing indicators left unscaled.

For each variant the code:

1. Chooses RF settings using inner forward CV only.
2. Fits a development forest and logistic reference, then evaluates October 3.
3. Calculates grouped permutation importance on that outer validation set.
4. Refits both models on the original final-refit rows before October 4 and saves them.

The candidate grid varies maximum depth, minimum leaf size and feature subsampling. The number of trees, class weights (`None`) and 0.5 classification threshold stay fixed. There is no feature deletion based on p-values.

Runtime depends on the real cohort and machine. Defaults require 80 CV forest fits (5 variants × 8 settings × 2 folds), plus development/final fits and permutation scoring. Reduce resources in notebook 01 before starting a new run if needed.

In [4]:
validation_metrics, selected_parameters = rf.fit_models(RUN_DIR, CONFIG)
display(validation_metrics[['model', 'variant', 'rows', 'roc_auc', 'log_loss',
                            'brier', 'pr_auc_average_precision', 'accuracy_threshold_0_5']])
display(pd.DataFrame(selected_parameters).T)

baseline: candidate 1/8
baseline: candidate 2/8
baseline: candidate 3/8
baseline: candidate 4/8
baseline: candidate 5/8
baseline: candidate 6/8
baseline: candidate 7/8
baseline: candidate 8/8
Completed baseline
expanding_3: candidate 1/8
expanding_3: candidate 2/8
expanding_3: candidate 3/8
expanding_3: candidate 4/8
expanding_3: candidate 5/8
expanding_3: candidate 6/8
expanding_3: candidate 7/8
expanding_3: candidate 8/8
Completed expanding_3
fixed24h_3: candidate 1/8
fixed24h_3: candidate 2/8
fixed24h_3: candidate 3/8
fixed24h_3: candidate 4/8
fixed24h_3: candidate 5/8
fixed24h_3: candidate 6/8
fixed24h_3: candidate 7/8
fixed24h_3: candidate 8/8
Completed fixed24h_3
expanding_5: candidate 1/8
expanding_5: candidate 2/8
expanding_5: candidate 3/8
expanding_5: candidate 4/8
expanding_5: candidate 5/8
expanding_5: candidate 6/8
expanding_5: candidate 7/8
expanding_5: candidate 8/8
Completed expanding_5
fixed24h_5: candidate 1/8
fixed24h_5: candidate 2/8
fixed24h_5: candidate 3/8
fixed2

## 3. Review importance for baseline AND history features

**Grouped permutation importance** measures the increase in validation log loss after shuffling a feature group. Larger positive values indicate more reliance by this fitted model. Negative values are possible. The standard deviation describes variation across shuffles, not a confidence interval or p-value.

All price terms and their missing flag move together, preserving `p²` and `p³`. Sine/cosine move together, as do each history ratio and its missing flag. An additional `ALL_HISTORY_TOGETHER` row evaluates reliance on the whole history block. This is a descriptive dependence perturbation, not the same operation as refitting without history; correlated features and event/time structure complicate interpretation.

RF impurity scores are exported separately as a training-based secondary measure. They can favor features with many possible split points; do not interpret them as significance tests. This workflow does not use XGBoost gain importance.

In [5]:
importance = pd.read_csv(RUN_DIR / 'validation_grouped_permutation_importance.csv')
for variant in ['baseline', 'fixed24h_5']:
    print()
    print(variant)
    display(importance.loc[importance.variant.eq(variant),
                          ['feature_group', 'mean_log_loss_increase', 'shuffle_sd', 'evaluated_rows']])
print('Models and selections are saved. Next: 03_rf_validation.ipynb.')


baseline
                      feature_group  ...  evaluated_rows
0  entry_price (p, p2, p3, missing)  ...            5000
1           market_log_observed_age  ...            5000
2                     entry_log_buy  ...            5000
3                market_log_wallets  ...            5000
4               entry_log_fill_rows  ...            5000
5              window_exposure_days  ...            5000
6               entry_concentration  ...            5000
7            hour_of_day (sin, cos)  ...            5000
8                market_log_dollars  ...            5000

[9 rows x 4 columns]

fixed24h_5
                            feature_group  ...  evaluated_rows
50       entry_price (p, p2, p3, missing)  ...            5000
51                market_log_observed_age  ...            5000
52                   ALL_HISTORY_TOGETHER  ...            5000
53                          entry_log_buy  ...            5000
54   history_avg_buy_size (incl. missing)  ...            5000
55      

## Method references

- [scikit-learn: RandomForestClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html)
- [scikit-learn: time-dependent cross-validation](https://scikit-learn.org/stable/modules/cross_validation.html#cross-validation-of-time-series-data)
- [scikit-learn: permutation importance and correlated features](https://scikit-learn.org/stable/modules/permutation_importance.html)

The custom split adds event purging and recorded-resolution maturity rules to chronological validation; ordinary `TimeSeriesSplit` alone would not enforce these project-specific checks.